In [ ]:
import pandas as pd
import numpy as np

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ---------------------------------------------------------
# FILE PATHS
# ---------------------------------------------------------

engagement_file = "Engagement_history_improved.csv"
hcp_file = "HCP_master_updated.csv"


# ---------------------------------------------------------
# LOAD DATASETS
# ---------------------------------------------------------

engagement_df = pd.read_csv(
    engagement_file,
    parse_dates=["engagement_date"]
)

hcp_df = pd.read_csv(hcp_file)


# ---------------------------------------------------------
# DISPLAY BASIC INFORMATION
# ---------------------------------------------------------

print("HCP Master Dataset")
print("------------------")
print("Shape:", hcp_df.shape)

print("\nEngagement History Dataset")
print("--------------------------")
print("Shape:", engagement_df.shape)

print("\nHCP Master Columns:")
print(hcp_df.columns.tolist())

print("\nEngagement Columns:")
print(engagement_df.columns.tolist())

HCP Master Dataset
------------------
Shape: (500, 12)

Engagement History Dataset
--------------------------
Shape: (11039, 4)

HCP Master Columns:
['hcp_id', 'first_name', 'last_name', 'specialty', 'segment', 'territory', 'city', 'state', 'practice_type', 'account_tenure', 'opt_out_flag', 'channel_preference']

Engagement Columns:
['hcp_id', 'engagement_date', 'channel', 'engagement_successful']


In [ ]:
# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

print("Missing values in HCP Master:")
display(hcp_df.isnull().sum())

print("\nMissing values in Engagement History:")
display(engagement_df.isnull().sum())


# ---------------------------------------------------------
# CHECK HCP ID COVERAGE
# ---------------------------------------------------------

master_ids = set(hcp_df["hcp_id"])
engagement_ids = set(engagement_df["hcp_id"])

missing_engagement_ids = master_ids - engagement_ids
orphan_engagement_ids = engagement_ids - master_ids

print("\nHCP IDs in Master Dataset:", len(master_ids))
print("HCP IDs in Engagement Dataset:", len(engagement_ids))

print("\nHCPs without engagement records:",
      len(missing_engagement_ids))

print("Engagement records without matching HCP:",
      len(orphan_engagement_ids))


# ---------------------------------------------------------
# CHECK DUPLICATES
# ---------------------------------------------------------

print("\nDuplicate rows in HCP Master:",
      hcp_df.duplicated().sum())

print("Duplicate rows in Engagement History:",
      engagement_df.duplicated().sum())


# ---------------------------------------------------------
# CHECK CHANNELS
# ---------------------------------------------------------

print("\nChannels available:")
display(
    engagement_df["channel"]
    .value_counts()
    .rename_axis("channel")
    .reset_index(name="interaction_count")
)

Missing values in HCP Master:


,0
hcp_id,0
first_name,0
last_name,0
specialty,0
segment,0
territory,0
city,0
state,0
practice_type,0
account_tenure,0



Missing values in Engagement History:


,0
hcp_id,0
engagement_date,0
channel,0
engagement_successful,0



HCP IDs in Master Dataset: 500
HCP IDs in Engagement Dataset: 500

HCPs without engagement records: 0
Engagement records without matching HCP: 0

Duplicate rows in HCP Master: 0
Duplicate rows in Engagement History: 0

Channels available:


,channel,interaction_count
0,email,2601
1,phone_call,2493
2,rep_visit,2326
3,digital_ad,1948
4,webinar,1671


In [ ]:
# ---------------------------------------------------------
# CHANNEL WEIGHTS
# ---------------------------------------------------------

CHANNEL_WEIGHTS = {
    "rep_visit": 0.30,
    "phone_call": 0.20,
    "webinar": 0.20,
    "email": 0.15,
    "digital_ad": 0.15
}


# ---------------------------------------------------------
# VALIDATE WEIGHTS
# ---------------------------------------------------------

weight_total = sum(CHANNEL_WEIGHTS.values())

print("Channel Weights:")
for channel, weight in CHANNEL_WEIGHTS.items():
    print(f"{channel:15s} -> {weight:.0%}")

print("\nTotal Weight:", weight_total)

assert np.isclose(weight_total, 1.0), \
    "Channel weights must add up to 1.0"

Channel Weights:
rep_visit       -> 30%
phone_call      -> 20%
webinar         -> 20%
email           -> 15%
digital_ad      -> 15%

Total Weight: 1.0


In [ ]:
# ---------------------------------------------------------
# REFERENCE DATE
# ---------------------------------------------------------

reference_date = engagement_df["engagement_date"].max()

print("Reference date:", reference_date)


# ---------------------------------------------------------
# AGGREGATE ENGAGEMENT DATA
# ---------------------------------------------------------

engagement_summary = (
    engagement_df
    .groupby(["hcp_id", "channel"], as_index=False)
    .agg(
        interaction_count=(
            "engagement_successful",
            "size"
        ),

        successful_interactions=(
            "engagement_successful",
            "sum"
        ),

        success_rate=(
            "engagement_successful",
            "mean"
        ),

        last_engagement_date=(
            "engagement_date",
            "max"
        )
    )
)


# ---------------------------------------------------------
# CALCULATE RECENCY
# ---------------------------------------------------------

engagement_summary["recency_days"] = (
    reference_date
    - engagement_summary["last_engagement_date"]
).dt.days


# ---------------------------------------------------------
# DISPLAY RESULT
# ---------------------------------------------------------

print("HCP × Channel Summary Shape:",
      engagement_summary.shape)

display(engagement_summary.head(10))

Reference date: 2026-01-02 17:00:00
HCP × Channel Summary Shape: (2274, 7)


,hcp_id,channel,interaction_count,successful_interactions,success_rate,last_engagement_date,recency_days
0,1,digital_ad,1,0,0.000000,2025-11-10 10:00:00,53
1,1,email,10,2,0.200000,2025-10-02 16:00:00,92
2,1,phone_call,4,0,0.000000,2025-10-16 11:00:00,78
3,1,rep_visit,3,0,0.000000,2025-12-15 08:00:00,18
4,1,webinar,5,2,0.400000,2025-12-26 10:00:00,7
5,2,digital_ad,2,0,0.000000,2025-07-23 09:00:00,163
6,2,email,3,0,0.000000,2025-09-08 16:00:00,116
7,2,phone_call,3,1,0.333333,2025-09-29 10:00:00,95
8,2,rep_visit,1,0,0.000000,2025-06-18 09:00:00,198
9,2,webinar,2,0,0.000000,2025-05-26 16:00:00,221


In [ ]:
# ---------------------------------------------------------
# PIVOT HCP × CHANNEL DATA
# ---------------------------------------------------------

channel_features = engagement_summary.pivot(
    index="hcp_id",
    columns="channel",
    values=[
        "interaction_count",
        "successful_interactions",
        "success_rate",
        "last_engagement_date",
        "recency_days"
    ]
)


# ---------------------------------------------------------
# FLATTEN MULTI-LEVEL COLUMN NAMES
# ---------------------------------------------------------

channel_features.columns = [
    f"{metric}_{channel}"
    for metric, channel in channel_features.columns
]

channel_features = channel_features.reset_index()


print("Pivoted Dataset Shape:",
      channel_features.shape)

display(channel_features.head())

Pivoted Dataset Shape: (500, 26)


,hcp_id,interaction_count_digital_ad,interaction_count_email,interaction_count_phone_call,interaction_count_rep_visit,interaction_count_webinar,successful_interactions_digital_ad,successful_interactions_email,successful_interactions_phone_call,successful_interactions_rep_visit,successful_interactions_webinar,success_rate_digital_ad,success_rate_email,success_rate_phone_call,success_rate_rep_visit,success_rate_webinar,last_engagement_date_digital_ad,last_engagement_date_email,last_engagement_date_phone_call,last_engagement_date_rep_visit,last_engagement_date_webinar,recency_days_digital_ad,recency_days_email,recency_days_phone_call,recency_days_rep_visit,recency_days_webinar
0,1,1,10,4,3,5,0,2,0,0,2,0.0,0.2,0.0,0.0,0.4,2025-11-10 10:00:00,2025-10-02 16:00:00,2025-10-16 11:00:00,2025-12-15 08:00:00,2025-12-26 10:00:00,53,92,78,18,7
1,2,2,3,3,1,2,0,0,1,0,0,0.0,0.0,0.333333,0.0,0.0,2025-07-23 09:00:00,2025-09-08 16:00:00,2025-09-29 10:00:00,2025-06-18 09:00:00,2025-05-26 16:00:00,163,116,95,198,221
2,3,15,7,3,2,3,0,0,0,1,0,0.0,0.0,0.0,0.5,0.0,2025-12-25 08:00:00,2025-09-01 10:00:00,2025-12-11 17:00:00,2025-11-19 13:00:00,2025-10-15 14:00:00,8,123,22,44,79
3,4,3,6,7,2,8,0,3,2,1,3,0.0,0.5,0.285714,0.5,0.375,2025-12-18 15:00:00,2025-11-05 09:00:00,2026-01-01 15:00:00,2025-10-22 11:00:00,2025-12-08 11:00:00,15,58,1,72,25
4,5,1,2,8,12,3,0,1,2,8,3,0.0,0.5,0.25,0.666667,1.0,2025-02-19 17:00:00,2025-08-07 11:00:00,2025-12-02 08:00:00,2026-01-02 17:00:00,2025-08-27 17:00:00,317,148,31,0,128


In [ ]:
# ---------------------------------------------------------
# ENSURE ALL REQUIRED CHANNEL COLUMNS EXIST
# ---------------------------------------------------------

channels = list(CHANNEL_WEIGHTS.keys())

for channel in channels:

    numeric_columns = [
        f"interaction_count_{channel}",
        f"successful_interactions_{channel}",
        f"success_rate_{channel}"
    ]

    for column in numeric_columns:

        if column not in channel_features.columns:
            channel_features[column] = 0

        channel_features[column] = (
            pd.to_numeric(
                channel_features[column],
                errors="coerce"
            )
            .fillna(0)
        )


    # Recency is handled separately
    recency_column = f"recency_days_{channel}"

    if recency_column not in channel_features.columns:
        channel_features[recency_column] = np.nan


print("All required channel columns are present.")

All required channel columns are present.


In [ ]:
# ---------------------------------------------------------
# CREATE FREQUENCY AND RECENCY SCORES
# ---------------------------------------------------------

for channel in channels:

    interaction_column = f"interaction_count_{channel}"
    success_column = f"success_rate_{channel}"
    recency_column = f"recency_days_{channel}"


    # -----------------------------------------------------
    # FREQUENCY SCORE
    # -----------------------------------------------------

    max_interactions = channel_features[
        interaction_column
    ].max()

    if max_interactions > 0:

        channel_features[
            f"frequency_score_{channel}"
        ] = (
            channel_features[interaction_column]
            / max_interactions
        )

    else:

        channel_features[
            f"frequency_score_{channel}"
        ] = 0


    # -----------------------------------------------------
    # RECENCY SCORE
    # -----------------------------------------------------

    recency_values = (
        channel_features[recency_column]
        .fillna(9999)
        .astype(float)
    )

    channel_features[
        f"recency_score_{channel}"
    ] = np.where(
        channel_features[interaction_column] > 0,
        1 / (1 + recency_values / 30),
        0
    )


print("Frequency and recency scores created.")

display(
    channel_features[
        [
            "hcp_id",
            "frequency_score_email",
            "recency_score_email",
            "frequency_score_rep_visit",
            "recency_score_rep_visit"
        ]
    ].head(10)
)

Frequency and recency scores created.


/tmp/ipykernel_2700/2855764909.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(9999)
/tmp/ipykernel_2700/2855764909.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(9999)
/tmp/ipykernel_2700/2855764909.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(9999)
/tmp/ipykernel_2700/2855764909.py:42: FutureWa

,hcp_id,frequency_score_email,recency_score_email,frequency_score_rep_visit,recency_score_rep_visit
0,1,0.588235,0.245902,0.176471,0.625000
1,2,0.176471,0.205479,0.058824,0.131579
2,3,0.411765,0.196078,0.117647,0.405405
3,4,0.352941,0.340909,0.117647,0.294118
4,5,0.117647,0.168539,0.705882,1.000000
5,6,0.058824,0.270270,0.294118,0.447761
6,7,0.117647,0.241935,0.941176,0.441176
7,8,0.647059,0.625000,0.058824,0.099010
8,9,0.411765,0.344828,0.235294,0.159574
9,10,0.294118,0.909091,0.117647,0.681818


In [ ]:
# ---------------------------------------------------------
# COMPONENT WEIGHTS
# ---------------------------------------------------------

SUCCESS_WEIGHT = 0.50
FREQUENCY_WEIGHT = 0.30
RECENCY_WEIGHT = 0.20


# ---------------------------------------------------------
# CREATE CHANNEL ENGAGEMENT SCORE
# ---------------------------------------------------------

for channel in channels:

    success_score = (
        channel_features[
            f"success_rate_{channel}"
        ]
        .fillna(0)
    )

    frequency_score = (
        channel_features[
            f"frequency_score_{channel}"
        ]
        .fillna(0)
    )

    recency_score = (
        channel_features[
            f"recency_score_{channel}"
        ]
        .fillna(0)
    )


    channel_features[
        f"channel_score_{channel}"
    ] = (
        SUCCESS_WEIGHT * success_score
        +
        FREQUENCY_WEIGHT * frequency_score
        +
        RECENCY_WEIGHT * recency_score
    )


# ---------------------------------------------------------
# DISPLAY CHANNEL SCORES
# ---------------------------------------------------------

score_columns = [
    "hcp_id"
] + [
    f"channel_score_{channel}"
    for channel in channels
]

display(
    channel_features[score_columns].head(10)
)

,hcp_id,channel_score_rep_visit,channel_score_phone_call,channel_score_webinar,channel_score_email,channel_score_digital_ad
0,1,0.177941,0.130556,0.512162,0.325651,0.091039
1,2,0.043963,0.270917,0.083904,0.094037,0.068588
2,3,0.366375,0.171635,0.145046,0.162745,0.439145
3,4,0.344118,0.467656,0.536591,0.424064,0.189583
4,5,0.745098,0.373361,0.627975,0.319002,0.036041
5,6,0.377788,0.268750,0.116906,0.071701,0.000000
6,7,0.745588,0.381556,0.412254,0.083681,0.068588
7,8,0.537449,0.175942,0.158462,0.455481,0.245014
8,9,0.477503,0.389382,0.124483,0.549638,0.302083
9,10,0.671658,0.097629,0.098182,0.270053,0.082580


In [ ]:
# ---------------------------------------------------------
# CALCULATE WEIGHTED ENGAGEMENT SCORE
# ---------------------------------------------------------

channel_features["weighted_engagement_score"] = 0.0


for channel, weight in CHANNEL_WEIGHTS.items():

    channel_features["weighted_engagement_score"] += (
        channel_features[
            f"channel_score_{channel}"
        ]
        * weight
    )


# ---------------------------------------------------------
# CONVERT TO 0-100 SCALE
# ---------------------------------------------------------

channel_features["weighted_engagement_score"] = (
    channel_features["weighted_engagement_score"]
    * 100
)


# ---------------------------------------------------------
# ROUND SCORE
# ---------------------------------------------------------

channel_features["weighted_engagement_score"] = (
    channel_features["weighted_engagement_score"]
    .round(2)
)


print("Weighted Engagement Score created.")

display(
    channel_features[
        ["hcp_id", "weighted_engagement_score"]
    ].head(10)
)

Weighted Engagement Score created.


,hcp_id,weighted_engagement_score
0,1,24.44
1,2,10.85
2,3,26.35
3,4,39.61
4,5,47.71
5,6,20.12
6,7,40.53
7,8,33.32
8,9,37.38
9,10,29.36


In [ ]:
# ---------------------------------------------------------
# CREATE ENGAGEMENT RANK
# ---------------------------------------------------------

channel_features["engagement_rank"] = (
    channel_features[
        "weighted_engagement_score"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# ---------------------------------------------------------
# CREATE ENGAGEMENT LEVEL
# ---------------------------------------------------------

channel_features["engagement_level"] = pd.cut(
    channel_features["weighted_engagement_score"],
    bins=[-np.inf, 20, 40, np.inf],
    labels=[
        "Low Engagement",
        "Medium Engagement",
        "High Engagement"
    ]
)


display(
    channel_features[
        [
            "hcp_id",
            "weighted_engagement_score",
            "engagement_rank",
            "engagement_level"
        ]
    ]
    .sort_values(
        "weighted_engagement_score",
        ascending=False
    )
    .head(20)
)

,hcp_id,weighted_engagement_score,engagement_rank,engagement_level
243,244,54.16,1,High Engagement
162,163,54.10,2,High Engagement
466,467,51.49,3,High Engagement
144,145,50.70,4,High Engagement
64,65,50.54,5,High Engagement
79,80,50.08,6,High Engagement
288,289,48.73,7,High Engagement
206,207,48.54,8,High Engagement
386,387,48.10,9,High Engagement
491,492,47.99,10,High Engagement


In [ ]:
# ---------------------------------------------------------
# MERGE HCP MASTER + ENGAGEMENT FEATURES
# ---------------------------------------------------------

final_dataset = hcp_df.merge(
    channel_features,
    on="hcp_id",
    how="left",
    validate="one_to_one"
)


print("Final Dataset Shape:",
      final_dataset.shape)

print("\nNumber of HCPs:",
      final_dataset["hcp_id"].nunique())

display(final_dataset.head())

Final Dataset Shape: (500, 55)

Number of HCPs: 500


,hcp_id,first_name,last_name,specialty,segment,territory,city,state,practice_type,account_tenure,opt_out_flag,channel_preference,interaction_count_digital_ad,interaction_count_email,interaction_count_phone_call,interaction_count_rep_visit,interaction_count_webinar,successful_interactions_digital_ad,successful_interactions_email,successful_interactions_phone_call,successful_interactions_rep_visit,successful_interactions_webinar,success_rate_digital_ad,success_rate_email,success_rate_phone_call,success_rate_rep_visit,success_rate_webinar,last_engagement_date_digital_ad,last_engagement_date_email,last_engagement_date_phone_call,last_engagement_date_rep_visit,last_engagement_date_webinar,recency_days_digital_ad,recency_days_email,recency_days_phone_call,recency_days_rep_visit,recency_days_webinar,frequency_score_rep_visit,recency_score_rep_visit,frequency_score_phone_call,recency_score_phone_call,frequency_score_webinar,recency_score_webinar,frequency_score_email,recency_score_email,frequency_score_digital_ad,recency_score_digital_ad,channel_score_rep_visit,channel_score_phone_call,channel_score_webinar,channel_score_email,channel_score_digital_ad,weighted_engagement_score,engagement_rank,engagement_level
0,1,Danielle,Johnson,endocrinology,medium_value,TERR_19,Nashville,TN,hospital,0.5,False,Digital-Heavy,1.0,10.0,4.0,3.0,5.0,0.0,2.0,0.0,0.0,2.0,0.0,0.2,0.000000,0.000000,0.400,2025-11-10 10:00:00,2025-10-02 16:00:00,2025-10-16 11:00:00,2025-12-15 08:00:00,2025-12-26 10:00:00,53,92,78,18,7,0.176471,0.625000,0.2500,0.277778,0.5,0.810811,0.588235,0.245902,0.0625,0.361446,0.177941,0.130556,0.512162,0.325651,0.091039,24.44,359,Medium Engagement
1,2,Joshua,Walker,cardiology,low_value,TERR_40,Detroit,MI,private_practice,3.3,False,In-Person-Heavy,2.0,3.0,3.0,1.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.333333,0.000000,0.000,2025-07-23 09:00:00,2025-09-08 16:00:00,2025-09-29 10:00:00,2025-06-18 09:00:00,2025-05-26 16:00:00,163,116,95,198,221,0.058824,0.131579,0.1875,0.240000,0.2,0.119522,0.176471,0.205479,0.1250,0.155440,0.043963,0.270917,0.083904,0.094037,0.068588,10.85,491,Low Engagement
2,3,Jill,Rhodes,cardiology,medium_value,TERR_21,Tucson,AZ,private_practice,0.8,False,Mixed,15.0,7.0,3.0,2.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.500000,0.000,2025-12-25 08:00:00,2025-09-01 10:00:00,2025-12-11 17:00:00,2025-11-19 13:00:00,2025-10-15 14:00:00,8,123,22,44,79,0.117647,0.405405,0.1875,0.576923,0.3,0.275229,0.411765,0.196078,0.9375,0.789474,0.366375,0.171635,0.145046,0.162745,0.439145,26.35,326,Medium Engagement
3,4,Patricia,Miller,endocrinology,high_value,TERR_25,Arlington,TX,clinic,5.6,False,Digital-Heavy,3.0,6.0,7.0,2.0,8.0,0.0,3.0,2.0,1.0,3.0,0.0,0.5,0.285714,0.500000,0.375,2025-12-18 15:00:00,2025-11-05 09:00:00,2026-01-01 15:00:00,2025-10-22 11:00:00,2025-12-08 11:00:00,15,58,1,72,25,0.117647,0.294118,0.4375,0.967742,0.8,0.545455,0.352941,0.340909,0.1875,0.666667,0.344118,0.467656,0.536591,0.424064,0.189583,39.61,102,Medium Engagement
4,5,Robert,Johnson,cardiology,low_value,TERR_47,Oakland,CA,hospital,2.7,False,Mixed,1.0,2.0,8.0,12.0,3.0,0.0,1.0,2.0,8.0,3.0,0.0,0.5,0.250000,0.666667,1.000,2025-02-19 17:00:00,2025-08-07 11:00:00,2025-12-02 08:00:00,2026-01-02 17:00:00,2025-08-27 17:00:00,317,148,31,0,128,0.705882,1.000000,0.5000,0.491803,0.3,0.189873,0.117647,0.168539,0.0625,0.086455,0.745098,0.373361,0.627975,0.319002,0.036041,47.71,12,High Engagement


In [ ]:
# ---------------------------------------------------------
# IDENTIFY BEST CHANNEL
# ---------------------------------------------------------

channel_score_columns = {
    channel: f"channel_score_{channel}"
    for channel in channels
}


def get_best_channel(row):

    scores = {
        channel: row[column]
        for channel, column
        in channel_score_columns.items()
    }

    return max(
        scores,
        key=scores.get
    )


final_dataset["recommended_channel"] = (
    final_dataset.apply(
        get_best_channel,
        axis=1
    )
)


display(
    final_dataset[
        [
            "hcp_id",
            "channel_preference",
            "weighted_engagement_score",
            "engagement_level",
            "recommended_channel"
        ]
    ].head(20)
)

,hcp_id,channel_preference,weighted_engagement_score,engagement_level,recommended_channel
0,1,Digital-Heavy,24.44,Medium Engagement,webinar
1,2,In-Person-Heavy,10.85,Low Engagement,phone_call
2,3,Mixed,26.35,Medium Engagement,digital_ad
3,4,Digital-Heavy,39.61,Medium Engagement,webinar
4,5,Mixed,47.71,High Engagement,rep_visit
5,6,Digital-Heavy,20.12,Medium Engagement,rep_visit
6,7,Mixed,40.53,High Engagement,rep_visit
7,8,Mixed,33.32,Medium Engagement,rep_visit
8,9,Digital-Heavy,37.38,Medium Engagement,email
9,10,In-Person-Heavy,29.36,Medium Engagement,rep_visit


In [ ]:
# ---------------------------------------------------------
# ENGAGEMENT ELIGIBILITY
# ---------------------------------------------------------

final_dataset["engagement_eligible"] = (
    ~final_dataset["opt_out_flag"]
)


# ---------------------------------------------------------
# PREVENT RECOMMENDATION FOR OPTED-OUT HCPs
# ---------------------------------------------------------

final_dataset.loc[
    final_dataset["opt_out_flag"] == True,
    "recommended_channel"
] = "Do Not Contact"


display(
    final_dataset[
        [
            "hcp_id",
            "opt_out_flag",
            "engagement_eligible",
            "recommended_channel"
        ]
    ].head(20)
)

,hcp_id,opt_out_flag,engagement_eligible,recommended_channel
0,1,False,True,webinar
1,2,False,True,phone_call
2,3,False,True,digital_ad
3,4,False,True,webinar
4,5,False,True,rep_visit
5,6,False,True,rep_visit
6,7,False,True,rep_visit
7,8,False,True,rep_visit
8,9,False,True,email
9,10,False,True,rep_visit


In [ ]:
# ---------------------------------------------------------
# IDENTIFY COLUMN GROUPS
# ---------------------------------------------------------

master_columns = hcp_df.columns.tolist()

engagement_columns = [
    col for col in final_dataset.columns
    if (
        col.startswith("interaction_count_")
        or col.startswith("successful_interactions_")
        or col.startswith("success_rate_")
        or col.startswith("last_engagement_date_")
        or col.startswith("recency_days_")
    )
]

score_columns = [
    col for col in final_dataset.columns
    if (
        col.startswith("frequency_score_")
        or col.startswith("recency_score_")
        or col.startswith("channel_score_")
    )
]

model_output_columns = [
    "weighted_engagement_score",
    "engagement_rank",
    "engagement_level",
    "recommended_channel",
    "engagement_eligible"
]


# ---------------------------------------------------------
# FINAL COLUMN ORDER
# ---------------------------------------------------------

final_columns = (
    master_columns
    + engagement_columns
    + score_columns
    + model_output_columns
)


final_dataset = final_dataset[
    final_columns
]


print("Final Dataset Shape:",
      final_dataset.shape)

display(final_dataset.head())

Final Dataset Shape: (500, 57)


,hcp_id,first_name,last_name,specialty,segment,territory,city,state,practice_type,account_tenure,opt_out_flag,channel_preference,interaction_count_digital_ad,interaction_count_email,interaction_count_phone_call,interaction_count_rep_visit,interaction_count_webinar,successful_interactions_digital_ad,successful_interactions_email,successful_interactions_phone_call,successful_interactions_rep_visit,successful_interactions_webinar,success_rate_digital_ad,success_rate_email,success_rate_phone_call,success_rate_rep_visit,success_rate_webinar,last_engagement_date_digital_ad,last_engagement_date_email,last_engagement_date_phone_call,last_engagement_date_rep_visit,last_engagement_date_webinar,recency_days_digital_ad,recency_days_email,recency_days_phone_call,recency_days_rep_visit,recency_days_webinar,frequency_score_rep_visit,recency_score_rep_visit,frequency_score_phone_call,recency_score_phone_call,frequency_score_webinar,recency_score_webinar,frequency_score_email,recency_score_email,frequency_score_digital_ad,recency_score_digital_ad,channel_score_rep_visit,channel_score_phone_call,channel_score_webinar,channel_score_email,channel_score_digital_ad,weighted_engagement_score,engagement_rank,engagement_level,recommended_channel,engagement_eligible
0,1,Danielle,Johnson,endocrinology,medium_value,TERR_19,Nashville,TN,hospital,0.5,False,Digital-Heavy,1.0,10.0,4.0,3.0,5.0,0.0,2.0,0.0,0.0,2.0,0.0,0.2,0.000000,0.000000,0.400,2025-11-10 10:00:00,2025-10-02 16:00:00,2025-10-16 11:00:00,2025-12-15 08:00:00,2025-12-26 10:00:00,53,92,78,18,7,0.176471,0.625000,0.2500,0.277778,0.5,0.810811,0.588235,0.245902,0.0625,0.361446,0.177941,0.130556,0.512162,0.325651,0.091039,24.44,359,Medium Engagement,webinar,True
1,2,Joshua,Walker,cardiology,low_value,TERR_40,Detroit,MI,private_practice,3.3,False,In-Person-Heavy,2.0,3.0,3.0,1.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.333333,0.000000,0.000,2025-07-23 09:00:00,2025-09-08 16:00:00,2025-09-29 10:00:00,2025-06-18 09:00:00,2025-05-26 16:00:00,163,116,95,198,221,0.058824,0.131579,0.1875,0.240000,0.2,0.119522,0.176471,0.205479,0.1250,0.155440,0.043963,0.270917,0.083904,0.094037,0.068588,10.85,491,Low Engagement,phone_call,True
2,3,Jill,Rhodes,cardiology,medium_value,TERR_21,Tucson,AZ,private_practice,0.8,False,Mixed,15.0,7.0,3.0,2.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.500000,0.000,2025-12-25 08:00:00,2025-09-01 10:00:00,2025-12-11 17:00:00,2025-11-19 13:00:00,2025-10-15 14:00:00,8,123,22,44,79,0.117647,0.405405,0.1875,0.576923,0.3,0.275229,0.411765,0.196078,0.9375,0.789474,0.366375,0.171635,0.145046,0.162745,0.439145,26.35,326,Medium Engagement,digital_ad,True
3,4,Patricia,Miller,endocrinology,high_value,TERR_25,Arlington,TX,clinic,5.6,False,Digital-Heavy,3.0,6.0,7.0,2.0,8.0,0.0,3.0,2.0,1.0,3.0,0.0,0.5,0.285714,0.500000,0.375,2025-12-18 15:00:00,2025-11-05 09:00:00,2026-01-01 15:00:00,2025-10-22 11:00:00,2025-12-08 11:00:00,15,58,1,72,25,0.117647,0.294118,0.4375,0.967742,0.8,0.545455,0.352941,0.340909,0.1875,0.666667,0.344118,0.467656,0.536591,0.424064,0.189583,39.61,102,Medium Engagement,webinar,True
4,5,Robert,Johnson,cardiology,low_value,TERR_47,Oakland,CA,hospital,2.7,False,Mixed,1.0,2.0,8.0,12.0,3.0,0.0,1.0,2.0,8.0,3.0,0.0,0.5,0.250000,0.666667,1.000,2025-02-19 17:00:00,2025-08-07 11:00:00,2025-12-02 08:00:00,2026-01-02 17:00:00,2025-08-27 17:00:00,317,148,31,0,128,0.705882,1.000000,0.5000,0.491803,0.3,0.189873,0.117647,0.168539,0.0625,0.086455,0.745098,0.373361,0.627975,0.319002,0.036041,47.71,12,High Engagement,rep_visit,True


In [ ]:
# ---------------------------------------------------------
# FINAL DATASET VALIDATION
# ---------------------------------------------------------

print("=" * 60)
print("FINAL DATASET VALIDATION")
print("=" * 60)

print("\nRows:", final_dataset.shape[0])
print("Columns:", final_dataset.shape[1])

print(
    "\nUnique HCPs:",
    final_dataset["hcp_id"].nunique()
)

print(
    "\nDuplicate HCP IDs:",
    final_dataset["hcp_id"].duplicated().sum()
)

print(
    "\nMissing HCP IDs:",
    final_dataset["hcp_id"].isna().sum()
)

print(
    "\nMissing Overall Scores:",
    final_dataset[
        "weighted_engagement_score"
    ].isna().sum()
)

print(
    "\nScore Statistics:"
)

display(
    final_dataset[
        "weighted_engagement_score"
    ].describe()
)

print("\nEngagement Level Distribution:")

display(
    final_dataset[
        "engagement_level"
    ]
    .value_counts()
    .sort_index()
)

FINAL DATASET VALIDATION

Rows: 500
Columns: 57

Unique HCPs: 500

Duplicate HCP IDs: 0

Missing HCP IDs: 0

Missing Overall Scores: 0

Score Statistics:


,weighted_engagement_score
count,500.000000
mean,30.547100
std,9.727044
min,5.190000
25%,23.537500
50%,30.410000
75%,38.052500
max,54.160000



Engagement Level Distribution:


,count
engagement_level,
Low Engagement,78
Medium Engagement,331
High Engagement,91


In [ ]:
# ---------------------------------------------------------
# SAVE FINAL DATASET
# ---------------------------------------------------------

output_file = "HCP_Weighted_Engagement_Final.csv"

final_dataset.to_csv(
    output_file,
    index=False
)

print(
    f"Final dataset saved successfully as: {output_file}"
)

Final dataset saved successfully as: HCP_Weighted_Engagement_Final.csv
